In [1]:
import pandas as pd
import numpy as np
import joblib
import tensorflow as tf
import ast
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Embedding, GRU, LSTM, Dense, Dropout, 
                                     Bidirectional, Conv1D, GlobalMaxPooling1D, 
                                     Flatten, Concatenate)
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from sklearn.metrics.pairwise import cosine_similarity
# Load and clean data
df = pd.read_csv("../all_data_cleaned.csv")
df.dropna(subset=["title", "supercategory_cleaned", "Bow", "weighted_rating_class_2"], inplace=True)
df = df.sample(n=100000, random_state=6).reset_index(drop=True)

C:\Users\kurt_\anaconda3\envs\pytorch-transformer\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
# Process labels
def list_to_str(x):
    if isinstance(x, str):
        try:
            lst = ast.literal_eval(x)
            return ' '.join(sorted(set(lst))) if isinstance(lst, list) else str(lst)
        except:
            return str(x)
    return str(x)

df["super_str"] = df["supercategory_cleaned"].apply(list_to_str)

# Label encoding
le_label = LabelEncoder()
df["y_label"] = le_label.fit_transform(df["super_str"])

In [3]:
# Clean text
def clean_text(text):
    if isinstance(text, str):
        words = text.lower().strip().split()
        return ' '.join(sorted(set(words)))
    return ""

df["clean_text"] = df["Bow"].apply(clean_text)

# Tokenization
tokenizer = Tokenizer(num_words=3000, oov_token="<OOV>")
tokenizer.fit_on_texts(df["clean_text"])
seq = tokenizer.texts_to_sequences(df["clean_text"])
padded_seq = pad_sequences(seq, maxlen=40, padding="post")

# Numeric data
scaler = MinMaxScaler()
numeric_data = scaler.fit_transform(df[["weighted_rating_class_2"]].astype(float))

# Prepare targets
y = to_categorical(df["y_label"])
num_classes = y.shape[1]

# Remove classes with only one sample
class_counts = pd.Series(np.argmax(y, axis=1)).value_counts()
single_sample_classes = class_counts[class_counts == 1].index
mask = ~np.isin(np.argmax(y, axis=1), single_sample_classes)

X_text = padded_seq[mask]
X_num = numeric_data[mask]
y = y[mask]
num_classes = y.shape[1]

# Train-test split
X_train_text, X_test_text, X_train_num, X_test_num, y_train, y_test = train_test_split(
    X_text, X_num, y, test_size=0.3, random_state=42, stratify=np.argmax(y, axis=1)
)

In [4]:
# Model architecture
def build_model(model_type="bigru"):
    input_text = Input(shape=(40,))
    x = Embedding(input_dim=3001, output_dim=300)(input_text)
    
    if model_type == "gru":
        x = GRU(128, return_sequences=True, dropout=0.3, recurrent_dropout=0.3)(x)
        x = GRU(128, dropout=0.3, recurrent_dropout=0.3)(x)
    elif model_type == "lstm":
        x = LSTM(128, return_sequences=True, dropout=0.3, recurrent_dropout=0.3)(x)
        x = LSTM(128, dropout=0.3, recurrent_dropout=0.3)(x)
    elif model_type == "bigru":
        x = Bidirectional(GRU(128, return_sequences=True, dropout=0.3, recurrent_dropout=0.3))(x)
        x = GlobalMaxPooling1D()(x)
    elif model_type == "bilstm":
        x = Bidirectional(LSTM(128, return_sequences=True, dropout=0.3, recurrent_dropout=0.3))(x)
        x = GlobalMaxPooling1D()(x)
    elif model_type == "cnn":
        x = Conv1D(128, 5, activation='relu')(x)
        x = GlobalMaxPooling1D()(x)
    elif model_type == "mlp":
        x = Flatten()(x)
    else:
        raise ValueError("Invalid model type")

    input_num = Input(shape=(1,))
    y = Dense(64, activation='relu')(input_num)

    z = Concatenate()([x, y])
    z = Dropout(0.5)(z)
    z = Dense(256, activation='relu')(z)
    output = Dense(num_classes, activation='softmax')(z)

    model = Model(inputs=[input_text, input_num], outputs=output)
    model.compile(optimizer=Adam(0.00025), 
                 loss="categorical_crossentropy", 
                 metrics=["accuracy"])
    return model


In [5]:
def train(model_type="bigru"):
    model = build_model(model_type)
    
    history = model.fit(
        [X_train_text, X_train_num], y_train,
        validation_data=([X_test_text, X_test_num], y_test),
        epochs=5, 
        batch_size=32,
        callbacks=[EarlyStopping(patience=3, restore_best_weights=True)]
    )

    model.save(f"{model_type}_supercategory_model.keras")
    joblib.dump(tokenizer, "tokenizer.pkl")
    joblib.dump(le_label, "le_label.pkl")
    joblib.dump(scaler, "scaler.pkl")

    y_pred = np.argmax(model.predict([X_test_text, X_test_num], verbose=0), axis=1)
    y_true = np.argmax(y_test, axis=1)
    
    print(classification_report(y_true, y_pred, 
                              target_names=le_label.classes_,
                              labels=np.arange(len(le_label.classes_)),
                              zero_division=0))
    print("F1 Score:", f1_score(y_true, y_pred, average="weighted"))
    print("Accuracy:", accuracy_score(y_true, y_pred))
    
    return history

In [6]:
def get_recommendations(user_text, rating_class, model_type="bigru", top_k=5):
    model = tf.keras.models.load_model(f"{model_type}_supercategory_model.keras")
    tokenizer = joblib.load("tokenizer.pkl")
    le_label = joblib.load("le_label.pkl")
    scaler = joblib.load("scaler.pkl")

    clean_input = ' '.join(sorted(set(user_text.lower().strip().split())))
    seq = tokenizer.texts_to_sequences([clean_input])
    padded = pad_sequences(seq, maxlen=40, padding="post")
    numeric_input = scaler.transform(np.array([[rating_class]]))

    pred_idx = np.argmax(model.predict([padded, numeric_input], verbose=0))
    pred_super = le_label.inverse_transform([pred_idx])[0]
    
    super_df = df[df["super_str"] == pred_super].copy()
    
    if len(super_df) == 0:
        return pred_super, None

    embed_model = Model(inputs=model.inputs, outputs=model.layers[-3].output)
    
    user_embedding = embed_model.predict([padded, numeric_input], verbose=0)
    
    seqs = tokenizer.texts_to_sequences(super_df["clean_text"])
    padded_seqs = pad_sequences(seqs, maxlen=40, padding="post")
    numeric_super = scaler.transform(super_df[["weighted_rating_class_2"]].values)
    prod_embeddings = embed_model.predict([padded_seqs, numeric_super], verbose=0)
    
    similarities = cosine_similarity(user_embedding, prod_embeddings)[0]
    super_df["similarity"] = similarities
    
    top_products = super_df.sort_values(by="similarity", ascending=False).head(top_k)
    
    return pred_super, top_products[["title", "price", "rating", "product_url", "similarity"]]

In [7]:
selected_model_type = "bilstm"
train(selected_model_type)

sample_input = "Araç torpido üstü aksesuarı"
pred_super, results = get_recommendations(
        user_text=sample_input,
        rating_class=1,
        model_type=selected_model_type,
        top_k=5
    )

print("Predicted Supercategory:", pred_super)
if results is not None:
    print("Top recommended products:")
    print(results.to_string(index=False))
else:
    print("No products found in this supercategory")

Epoch 1/5
2188/2188 [==============================] - 732s 333ms/step - loss: 2.0364 - accuracy: 0.5758 - val_loss: 0.6462 - val_accuracy: 0.8754
Epoch 2/5
2188/2188 [==============================] - 730s 334ms/step - loss: 0.4643 - accuracy: 0.9028 - val_loss: 0.2372 - val_accuracy: 0.9515
Epoch 3/5
2188/2188 [==============================] - 723s 330ms/step - loss: 0.2199 - accuracy: 0.9496 - val_loss: 0.1275 - val_accuracy: 0.9742
Epoch 4/5
2188/2188 [==============================] - 713s 326ms/step - loss: 0.1313 - accuracy: 0.9696 - val_loss: 0.0827 - val_accuracy: 0.9823
Epoch 5/5
2188/2188 [==============================] - 711s 325ms/step - loss: 0.0888 - accuracy: 0.9784 - val_loss: 0.0693 - val_accuracy: 0.9848
                                 precision    recall  f1-score   support

                         abajur       0.84      0.89      0.86        18
            acma alt bebek bezi       0.00      0.00      0.00         0
                       aksesuar       0.99   

C:\Users\kurt_\anaconda3\envs\pytorch-transformer\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


Predicted Supercategory: tablet
Top recommended products:
                                                                    title   price  rating                                                                                                                      product_url  similarity
                                ['pad', 'xiaomi', 'turkiye', 'garantili'] 17619.0     4.5                                     https://www.hepsiburada.com/xiaomi-pad-7-8gb-256gb-xiaomi-turkiye-garantili-p-HBCV000082I15M    0.866278
                    ['galaxy', 'tab', 'wifi', 'tablet', 'mystic', 'pink'] 22990.0     4.8                          https://www.hepsiburada.com/samsung-galaxy-tab-s7-fe-64gb-12-4-wifi-tablet-mystic-pink-p-HBCV00000SGHS4    0.862516
                          ['ipad', 'nesil', 'wifi', 'cellular', 'tablet'] 22999.0     4.9                        https://www.hepsiburada.com/apple-ipad-8-nesil-128-gb-10-2-wifi-cellular-tablet-myml2tu-a-pm-HB00000XAMII    0.856958
          ['ipad',

C:\Users\kurt_\anaconda3\envs\pytorch-transformer\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


In [8]:
sample_input = "Bebek"
pred_super, results = get_recommendations(
        user_text=sample_input,
        rating_class=1,
        model_type=selected_model_type,
        top_k=5
    )

print("Predicted Supercategory:", pred_super)
if results is not None:
    print("Top recommended products:")
    print(results.to_string(index=False))
else:
    print("No products found in this supercategory")

C:\Users\kurt_\anaconda3\envs\pytorch-transformer\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


C:\Users\kurt_\anaconda3\envs\pytorch-transformer\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


Predicted Supercategory: bebek
Top recommended products:
                                           title  price  rating                                                                                          product_url  similarity
                  ['kapitone', 'bebek', 'yelek'] 229.99     4.1                            https://www.hepsiburada.com/messido-kapitone-bebek-yelek-p-HBCV00005O2IW2    0.984383
                               ['tayt', 'bebek'] 129.99     5.0                            https://www.hepsiburada.com/hellobaby-kiz-tayt-kiz-bebek-p-HBCV00007DHKD0    0.977196
                               ['tayt', 'bebek']  77.99     5.0                            https://www.hepsiburada.com/hellobaby-kiz-tayt-kiz-bebek-p-HBCV00007PVZAB    0.977196
                             ['cizgili', 'tayt'] 337.79     4.3                                      https://www.hepsiburada.com/koton-cizgili-tayt-pm-HBC00000NRK5C    0.974016
['lisansli', 'beyaz', 'bebek', 'corap', 'takim'] 249.00   

In [9]:
sample_input = "Oto aksesuar"
pred_super, results = get_recommendations(
        user_text=sample_input,
        rating_class=1,
        model_type=selected_model_type,
        top_k=5
    )

print("Predicted Supercategory:", pred_super)
if results is not None:
    print("Top recommended products:")
    print(results.to_string(index=False))
else:
    print("No products found in this supercategory")

C:\Users\kurt_\anaconda3\envs\pytorch-transformer\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
C:\Users\kurt_\anaconda3\envs\pytorch-transformer\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


Predicted Supercategory: aksesuar oto
Top recommended products:
                         title  price  rating                                                                                                  product_url  similarity
                       ['rag']  980.0     5.0                                             https://www.hepsiburada.com/auto-finesse-rag-top-pm-HB00000AHQU7    0.992790
             ['motor', 'yagi']  855.0     5.0 https://www.hepsiburada.com/mobil-10w40-motor-yagi-4-l-uretim-yili-2023-gojo-yazlik-camsuyu-pm-HBC00001HGWID    0.990378
                            []  285.0     5.0                             https://www.hepsiburada.com/castrol-castrolpower110w-40-4t-2023-pm-HBC000040DNXN    0.990378
['seramik', 'katkili', 'cila']  249.0     4.3                             https://www.hepsiburada.com/magicfox-seramik-katkili-hizli-cila-pm-HBC00004UURS2    0.986822
 ['seramik', 'etkili', 'cila']  265.0     4.0                                https://www.hepsiburada.

In [10]:
sample_input = "Iphone cep telefonu aksesuarı telefon tutacağı"
pred_super, results = get_recommendations(
        user_text=sample_input,
        rating_class=1,
        model_type=selected_model_type,
        top_k=5
    )

print("Predicted Supercategory:", pred_super)
if results is not None:
    print("Top recommended products:")
    print(results.to_string(index=False))
else:
    print("No products found in this supercategory")

C:\Users\kurt_\anaconda3\envs\pytorch-transformer\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
C:\Users\kurt_\anaconda3\envs\pytorch-transformer\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


Predicted Supercategory: aksesuarlari cep telefonu
Top recommended products:
                                      title  price  rating                                                                               product_url  similarity
          ['sanal', 'gerceklik', 'gozlugu']  649.0     5.0               https://www.hepsiburada.com/vr-pro2-sanal-gerceklik-gozlugu-pm-HB000003WHG2    0.967135
['slink', 'mini', 'hdmim', 'hdmim', 'gold']  152.1     3.8 https://www.hepsiburada.com/s-link-sl-hm55-3mt-mini-hdmi-m-to-hdmi-m-gold-pm-HB000001EF8Y    0.966834
                                         []  999.0     5.0                              https://www.hepsiburada.com/soultech-bh050b-pm-HBC00004EB837    0.960661
                                   ['pods']  835.0     4.4                     https://www.hepsiburada.com/general-mobile-gm-pods-3-pm-HBC00004HS7OB    0.960617
                      ['freebuds', 'siyah'] 3217.0     4.7                     https://www.hepsiburada.com/huawei-free

In [11]:
sample_input = "Bilgisayar"
pred_super, results = get_recommendations(
        user_text=sample_input,
        rating_class=1,
        model_type=selected_model_type,
        top_k=5
    )

print("Predicted Supercategory:", pred_super)
if results is not None:
    print("Top recommended products:")
    print(results.to_string(index=False))
else:
    print("No products found in this supercategory")

C:\Users\kurt_\anaconda3\envs\pytorch-transformer\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


Predicted Supercategory: dekoratif hediyeler
Top recommended products:
                                                        title  price  rating                                                                                            product_url  similarity
                      ['anime', 'baskili', 'termos', 'model'] 728.28     0.0                              https://www.hepsiburada.com/anime-baskili-termos-model-3-pm-HBC00009CO6X2    0.868276
                ['nala', 'dekoratif', 'aski', 'naturelbeyaz'] 747.90     5.0 https://www.hepsiburada.com/bella-maison-nala-dekoratif-aski-naturel-beyaz-38x5x18-cm-pm-HBC00007F1RUF    0.861224
['rengarenk', 'rainboow', 'metal', 'dilek', 'ruzgar', 'cani'] 449.90     4.8   https://www.hepsiburada.com/hediye-fikri-rengarenk-rainboow-metal-dilek-ruzgar-cani-pm-HBC00004I9U8Z    0.860197
                     ['tavuskusu', 'ruzgar', 'cani', 'model'] 367.17     0.0                             https://www.hepsiburada.com/tavuskusu-ruzgar-cani-model-

C:\Users\kurt_\anaconda3\envs\pytorch-transformer\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


In [12]:
sample_input = "Erkek alt giyim pantolon"
pred_super, results = get_recommendations(
        user_text=sample_input,
        rating_class=1,
        model_type=selected_model_type,
        top_k=5
    )

print("Predicted Supercategory:", pred_super)
if results is not None:
    print("Top recommended products:")
    print(results.to_string(index=False))
else:
    print("No products found in this supercategory")

C:\Users\kurt_\anaconda3\envs\pytorch-transformer\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


C:\Users\kurt_\anaconda3\envs\pytorch-transformer\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


Predicted Supercategory: giyim
Top recommended products:
                                                              title   price  rating                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            

In [18]:
sample_input = "Kadın üst giyim t-shirt"
pred_super, results = get_recommendations(
        user_text=sample_input,
        rating_class=1,
        model_type=selected_model_type,
        top_k=5
    )

print("Predicted Supercategory:", pred_super)
if results is not None:
    print("Top recommended products:")
    print(results.to_string(index=False))
else:
    print("No products found in this supercategory")

C:\Users\kurt_\anaconda3\envs\pytorch-transformer\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
C:\Users\kurt_\anaconda3\envs\pytorch-transformer\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


Predicted Supercategory: giyim
Top recommended products:
                                                      title   price  rating                                                                                     product_url  similarity
            ['yerli', 'yagmurluk', 'takimbalikci', 'takim']  479.74     5.0           https://www.hepsiburada.com/hegi-yerli-yagmurluk-takim-balikci-takim-pm-HBC00006VSHL2    0.989868
                                   ['parasut', 'pantolonu']  699.99     5.0                            https://www.hepsiburada.com/mango-parasut-pantolonu-p-HBCV000069X586    0.986321
['prestijboncuk', 'trendy', 'tarlatan', 'korsed', 'series']  434.54     5.0 https://www.hepsiburada.com/prestijboncuk-trendy-tarlatan-korsed-series-ks-427-pm-HBC00002U8UHN    0.984723
['prestijboncuk', 'trendy', 'tarlatan', 'trendy', 'series']  387.96     3.0 https://www.hepsiburada.com/prestijboncuk-trendy-tarlatan-trendy-series-tr-234-pm-HBC00002U8Y9N    0.981902
                       